In [1]:
!pip install -q "unsloth[colab-new]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.3/76.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 M

In [2]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("BF16 supported:", torch.cuda.is_bf16_supported())


GPU: Tesla T4
CUDA: 12.8
BF16 supported: True


In [3]:
from unsloth import FastLanguageModel
from datasets import load_dataset
import torch

print("Imports successful!")

# Load BillSum dataset
train_data = load_dataset("FiscalNote/billsum", split="train")
test_data = load_dataset("FiscalNote/billsum", split="test")

print("Train examples:", len(train_data))
print("Test examples:", len(test_data))
print("Columns:", train_data.column_names)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Imports successful!


README.md:   0%|          | 0.00/7.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 91.8MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.8MB            

...

data/ca_test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18949 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

Train examples: 18949
Test examples: 3269
Columns: ['text', 'summary', 'title']


In [4]:
# Use a manageable subset for training
train_data = train_data.select(range(2000))

print("Training examples used:", len(train_data))
print("Test examples:", len(test_data))

Training examples used: 2000
Test examples: 3269


In [5]:
from unsloth import FastLanguageModel
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)

print("Model loaded successfully!")
print("Model:", model_name)

==((====))==  Unsloth 2026.8.19: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully!
Model: Qwen/Qwen2.5-1.5B-Instruct


In [6]:
legal_prompt = """You are a legal document summarization assistant.

Summarize the following legislative bill clearly and concisely.

### Legislative Bill:
{}

### Summary:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    bills = examples["text"]
    summaries = examples["summary"]

    texts = []

    for bill, summary in zip(bills, summaries):
        text = legal_prompt.format(bill, summary) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}

formatted_train = train_data.map(
    formatting_prompts_func,
    batched=True
)

print(formatted_train[0]["text"][:2000])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

You are a legal document summarization assistant.

Summarize the following legislative bill clearly and concisely.

### Legislative Bill:
SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES 
              TO NONPROFIT ORGANIZATIONS.

    (a) Definitions.--In this section:
            (1) Business entity.--The term ``business entity'' means a 
        firm, corporation, association, partnership, consortium, joint 
        venture, or other form of enterprise.
            (2) Facility.--The term ``facility'' means any real 
        property, including any building, improvement, or appurtenance.
            (3) Gross negligence.--The term ``gross negligence'' means 
        voluntary and conscious conduct by a person with knowledge (at 
        the time of the conduct) that the conduct is likely to be 
        harmful to the health or well-being of another person.
            (4) Intentional misconduct.--The term ``intentional 
        misconduct'' means conduct by a per

In [7]:

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=False,
    random_state=42,
)

model.print_trainable_parameters()

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.8.19 patched 28 layers with 28 QKV layers, 28 O layers and 0 MLP layers.


trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_train,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=False,

    args=TrainingArguments(
        output_dir="./legal_lora",
        num_train_epochs=1,

        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        learning_rate=2e-4,
        warmup_steps=5,
        logging_steps=10,

        # Unsloth detected this T4 setup should use FP16
        fp16=True,
        bf16=False,

        optim="adamw_8bit",

        weight_decay=0.01,
        lr_scheduler_type="linear",

        save_strategy="no",
        report_to="none",

        seed=42,
    ),
)

print("Trainer is ready!")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2000 [00:00<?, ? examples/s]

Trainer is ready!


In [9]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,358,144 of 1,548,072,448 (0.28% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.405678
20,1.264285
30,1.267749
40,1.198145
50,1.241794
60,1.173936
70,1.191209
80,1.202786
90,1.201499
100,1.219662


In [10]:
def create_short_examples(examples):
    texts = []

    for bill, summary in zip(examples["text"], examples["summary"]):

        # Keep enough room for the target summary
        bill_tokens = tokenizer(
            bill,
            truncation=True,
            max_length=700,
            add_special_tokens=False
        )["input_ids"]

        summary_tokens = tokenizer(
            summary,
            truncation=True,
            max_length=250,
            add_special_tokens=False
        )["input_ids"]

        short_bill = tokenizer.decode(
            bill_tokens,
            skip_special_tokens=True
        )

        short_summary = tokenizer.decode(
            summary_tokens,
            skip_special_tokens=True
        )

        text = f"""You are a legal document summarization assistant.

Summarize the following legislative bill clearly and concisely.

### Legislative Bill:
{short_bill}

### Summary:
{short_summary}{tokenizer.eos_token}"""

        texts.append(text)

    return {"text": texts}


corrected_train = train_data.map(
    create_short_examples,
    batched=True
)

print(corrected_train[0]["text"][:3000])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

You are a legal document summarization assistant.

Summarize the following legislative bill clearly and concisely.

### Legislative Bill:
SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES 
              TO NONPROFIT ORGANIZATIONS.

    (a) Definitions.--In this section:
            (1) Business entity.--The term ``business entity'' means a 
        firm, corporation, association, partnership, consortium, joint 
        venture, or other form of enterprise.
            (2) Facility.--The term ``facility'' means any real 
        property, including any building, improvement, or appurtenance.
            (3) Gross negligence.--The term ``gross negligence'' means 
        voluntary and conscious conduct by a person with knowledge (at 
        the time of the conduct) that the conduct is likely to be 
        harmful to the health or well-being of another person.
            (4) Intentional misconduct.--The term ``intentional 
        misconduct'' means conduct by a per

In [11]:
print("===== END OF FORMATTED EXAMPLE =====")
print(corrected_train[0]["text"][-1200:])

===== END OF FORMATTED EXAMPLE =====
ity of that entity in connection with a use of such facility by a nonprofit organization if: (1) the use occurs outside the scope of business of the business entity; (2) such injury or death occurs during a period that such facility is used by such organization; and (3) the business entity authorized the use of such facility by the organization. 
Makes this Act inapplicable to an injury or death that results from an act or omission of a business entity that constitutes gross negligence or intentional misconduct, including misconduct that: (1) constitutes a hate crime or a crime of violence or act of international terrorism for which the defendant has been convicted in any court; or (2) involves a sexual offense for which the defendant has been convicted in any court or misconduct for which the defendant has been found to have violated a Federal or State civil rights law. 
Preempts State laws to the extent that such laws are inconsistent with this Ac

In [12]:
# Save the trained LoRA adapter

model.save_pretrained("./legal_qwen_lora")
tokenizer.save_pretrained("./legal_qwen_lora")

print("LoRA adapter saved successfully!")

Unsloth: Restored added_tokens_decoder metadata in ./legal_qwen_lora/tokenizer_config.json.


LoRA adapter saved successfully!


In [14]:
# Find the shortest unseen test example

short_test = min(
    test_data,
    key=lambda x: len(x["text"])
)

print("Bill characters:", len(short_test["text"]))
print("\nReference summary:")
print(short_test["summary"])

Bill characters: 5004

Reference summary:
Requires each State to submit to the Attorney General a written report (and subsequent updates) that specifies each location at which a State law enforcement agency stores explosive materials that have been transported in interstate or foreign commerce and the types and amounts of such materials. Directs the Attorney General to prescribe final regulations governing the storage of such materials by such agencies, including public safety and security standards and requirements for video surveillance or an alarm system.

Authorizes the Attorney General to: (1) make matching grants to State and local law enforcement agencies for complying with such regulations; (2) enter any place where such an agency keeps such explosive materials for inspection and compliance determinations; and (3) reduce by ten percent the funds that an agency would otherwise receive under any Department of Justice grant program if the agency fails to comply.


In [15]:
# Prepare the unseen bill using the same input length strategy
bill_tokens = tokenizer(
    short_test["text"],
    truncation=True,
    max_length=700,
    add_special_tokens=False
)["input_ids"]

short_bill = tokenizer.decode(
    bill_tokens,
    skip_special_tokens=True
)

prompt = f"""You are a legal document summarization assistant.

Summarize the following legislative bill clearly and concisely.

### Legislative Bill:
{short_bill}

### Summary:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

generated_summary = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("=" * 70)
print("GENERATED SUMMARY")
print("=" * 70)
print(generated_summary.strip())

print("\n" + "=" * 70)
print("REFERENCE SUMMARY")
print("=" * 70)
print(short_test["summary"])

Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED SUMMARY
This legislation requires States and local law enforcement agencies to 
report locations, types, and amounts of explosive materials they store 
and keep. It also requires the Attorney General to prescribe regulations 
that specify standards of public safety and security against theft 
which any place storing or keeping explosive materials must meet. The 
regulations should include requirements for video surveillance or an 
alarm system capable of notifying the agency if unauthorized entry is 
attempted. The Attorney General may enter into business hours to inspect 
the explosive materials and determine whether the materials are being 
stored or kept in compliance with the regulations. If a

REFERENCE SUMMARY
Requires each State to submit to the Attorney General a written report (and subsequent updates) that specifies each location at which a State law enforcement agency stores explosive materials that have been transported in interstate or foreign commerce and the typ

# LegalDoc-PEFT: Legislative Summarization using LoRA

## 1. Problem Statement

This project demonstrates domain-specific fine-tuning of a general-purpose
language model for legal document summarization.

The selected task is legislative bill summarization, where the model receives
a legislative bill and generates a concise summary of its key provisions.

## 2. Dataset

The **BillSum (FiscalNote/billsum)** dataset was used.

- Training examples available: 18,949
- Test examples available: 3,269
- Training examples used: 2,000
- Input: Legislative bill text
- Target: Human-written legislative summary

The training examples were formatted as an instruction-following task in
which the model was instructed to summarize the legislative bill.

## 3. Base Model

The base model used was **Qwen2.5-1.5B-Instruct**.

The model was loaded using 4-bit quantization to reduce GPU memory usage and
make parameter-efficient fine-tuning practical on a Tesla T4 GPU.

## 4. LoRA / PEFT Configuration

| Parameter | Configuration |
|---|---|
| Base model | Qwen2.5-1.5B-Instruct |
| Quantization | 4-bit |
| LoRA rank (r) | 16 |
| LoRA alpha | 32 |
| LoRA dropout | 0 |
| Target modules | q_proj, k_proj, v_proj, o_proj |
| Total parameters | 1,548,072,448 |
| Trainable parameters | 4,358,144 |
| Trainable percentage | 0.2815% |

Only a very small fraction of the model parameters were updated during
fine-tuning, demonstrating the parameter efficiency of LoRA.

## 5. Training Configuration

- Training examples: 2,000
- Number of epochs: 1
- Training steps: 250
- Batch size per device: 2
- Gradient accumulation steps: 4
- Effective batch size: 8
- Learning rate: 2 × 10^-4
- Maximum sequence length: 1024
- Optimizer: AdamW 8-bit
- Training precision: FP16
- GPU: NVIDIA Tesla T4

## 6. Training Results

The model completed all 250 training steps successfully.

The recorded training loss decreased from approximately **1.406** at
step 10 to approximately **1.175** at the final step.

This indicates that the LoRA adapters learned from the legal summarization
training examples during the fine-tuning process.

## 7. Model Features

The resulting domain-adapted model provides:

- Legislative bill summarization
- Legal-domain instruction following
- Parameter-efficient fine-tuning through LoRA
- 4-bit quantized model loading
- Lightweight LoRA adapter that can be stored separately from the base model
- Ability to reuse the base model with or without the legal-domain adapter

## 8. Evaluation

The trained model was tested on an unseen example from the BillSum test set.
The generated summary was compared qualitatively with the reference summary
provided by the dataset.

### Generated Summary

Paste the generated summary from the inference cell here.

### Reference Summary

Paste the corresponding BillSum reference summary here.

## 9. Limitations

This is a proof-of-concept domain adaptation experiment.

The model was trained on only 2,000 examples for one epoch, and the maximum
sequence length was limited to 1024 tokens. Therefore, the experiment should
not be interpreted as a production-ready legal summarization system.

The model is intended for summarization assistance and should not be used as
a substitute for professional legal review.

## 10. Conclusion

This experiment demonstrates how LoRA-based Parameter-Efficient Fine-Tuning
can adapt a general-purpose language model to a specialized legal task.

By updating only approximately **0.28% of the model parameters**, the model
was adapted toward legislative document summarization while keeping the
computational and memory requirements substantially lower than full
fine-tuning.